In [1]:
import os
model_name = os.getenv("model_name")

In [2]:
from validators import validate_inputs

user_input = {
 "customer_name": "Amit",
 "customer_type": "Premium",
 "ticket_subject": "Charged twice and no response from support",
 "ticket_body": "Hi team, I cancelled my premium subscription last month, but I was still charged again this month. I also noticed that the same invoice amount appears twice on my bank statement. I contacted support two times last week but have not received any proper response. This is extremely frustrating. If this is not resolved today, I will escalate this publicly on LinkedIn and also ask our finance team to block future payments. Please refund the incorrect charge immediately. Regards, Amit",
 "product_area": "Billing and subscription",
 "previous_interaction_history": "Customer says they contacted support two times last week and did not receive a proper response.",
 "sla_tier": "Premium",
 "response_tone": "Professional and empathetic",
 "business_rules": [
 "Do not promise refund before verification.",
 "Do not confirm cancellation unless verified.",
 "Ask for invoice ID or registered account email if required.",
 "Escalate premium customer billing issues to billing support."
 ]
}

validate_inputs(user_input)
print("All inputs are valid.")

All inputs are valid.


In [3]:
import os
from prompts import System_message, assistant_message_role,assistant_message_zero_shot, assistant_message_few_shot
prompt_tamplate = [
    {"role": "system", "content": System_message},
    {"role": "assistant", "content": assistant_message_role},
    {"role": "assistant", "content": assistant_message_zero_shot},
    {"role": "assistant", "content": assistant_message_few_shot},
    {"role": "user", "content": str(user_input)}
]


In [8]:
from groq_client import client
def call_groq_model(promtps, model=model_name, temperature=0.3):
    response = client.chat.completions.create(
        model=model,
        messages=promtps,
        temperature=temperature,
    )
    return response.choices[0].message.content



In [9]:
from groq_client import client
from output_parser import save_json_output

def call_groq_model(promtps, model=model_name, temperature=0.3):
    response = client.chat.completions.create(
        model=model,
        messages=promtps,
        temperature=temperature,
    )
    return response.choices[0].message.content


def save_response():
    llm_output = call_groq_model(prompt_tamplate)
    print(f"LLM Response:\n{llm_output}\n")
    try:
        saved_path = save_json_output(llm_output, filename="sample_ticket_output.json")
        print(f"Stored successfully: {saved_path}")
    except Exception as e:
        print(f"Failed to store JSON output: {type(e).__name__}: {e}")
        raise


save_response()


LLM Response:
```
{
  "ticket_summary": {
    "short_summary": "Premium customer Amit was charged twice after cancelling subscription and did not receive a proper response from support.",
    "customer_problem": "Double charge after subscription cancellation",
    "business_impact": "Potential loss of customer and public escalation",
    "customer_requested_action": "Refund the incorrect charge immediately",
    "important_context": ["Previous interactions with support", "Cancellation of premium subscription"],
    "missing_information": ["Invoice ID or registered account email"]
  },
  "classification": {
    "primary_category": "BILLING_ISSUE",
    "secondary_categories": ["ACCOUNT_ACCESS"],
    "category_reasoning_summary": "The customer is experiencing a billing issue due to a double charge after cancelling their subscription.",
    "confidence_score": 0.9
  },
  "sentiment_analysis": {
    "sentiment": "FRUSTRATED",
    "emotion_signals": ["extremely frustrating", "public escalati